In [ ]:
!pip install -U transformers --quiet

In [ ]:
seed_tasks = [
    {
        "instruction": "Translate the sentence into French.",
        "input": "I am happy.",
        "output": "Je suis heureux."
    },
    {
        "instruction": "Summarize the paragraph.",
        "input": "The cat chased the mouse into the attic. After a long game of hide and seek, the mouse escaped.",
        "output": "The cat chased the mouse, but it got away."
    },
    {
        "instruction": "Convert the temperature from Fahrenheit to Celsius.",
        "input": "98.6",
        "output": "37.0"
    },
    {
        "instruction": "Write a tweet promoting climate awareness.",
        "input": "",
        "output": "Small actions make a big difference. Switch off lights, cut down plastic, and support clean energy! #ClimateAction"
    },
    {
        "instruction": "Classify the sentiment of the review.",
        "input": "The product broke after one day. Totally useless.",
        "output": "Negative"
    }
]


In [ ]:
pip install transformers datasets torch


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import random

seed_instructions = [
    "Translate the sentence into French.",
    "Summarize the paragraph.",
    "Convert the temperature from Fahrenheit to Celsius.",
    "Write a tweet promoting climate awareness.",
    "Classify the sentiment of the review.",
]

def generate_instruction():
    seed = random.choice(seed_instructions)
    prompt = (
        f"Here is an example instruction:\n{seed}\n"
        f"Now generate a different but useful instruction task in natural language:\nInstruction:"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(**inputs, max_length=64, do_sample=True, top_p=0.92, temperature=0.9)
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
for _ in range(5):
    print(generate_instruction())


To convert the temperature to Celsius, you need to increase the temperature to minus 6 degrees.
@PattooClutch You must be kidding me.
Place the item on the table.
If you're looking to make a video game console out of a pc, you may want to add a little game controller to your existing computer.
@BoiaSam aww i want to know what ya'll do if i can be part of a climate change project


In [ ]:
def generate_example(instruction, mode="input_first"):
    if mode == "input_first":
        example_prompt = (
            f"Instruction: {instruction}\n"
            f"Input: [Write a plausible input for this instruction and give a correct output.]\n"
            f"Output:"
        )
    else:
        example_prompt = f"Instruction: {instruction}\nGive an example input and output."

    inputs = tokenizer(example_prompt, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_length=128, do_sample=True, top_p=0.95, temperature=0.9)
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
def clean_task(instruction, raw_example):
    if not instruction.strip():
        return None

    if len(instruction.strip()) < 15 or len(raw_example.strip()) < 15:
        return None

    if "Instruction:" in instruction or instruction.count(" ") < 3:
        return None

    return {"instruction": instruction.strip(), "raw_example": raw_example.strip()}


In [ ]:
new_tasks = []
for _ in range(20):
    instr = generate_instruction()
    io = generate_example(instr)
    cleaned = clean_task(instr, io)
    if cleaned:
        new_tasks.append(cleaned)


In [ ]:
!pip install transformers datasets --quiet

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


In [ ]:
examples = [
    {
        "instruction": "Translate the sentence into Spanish.",
        "input": "I am happy.",
        "output": "Estoy feliz."
    },
    {
        "instruction": "Summarize the paragraph.",
        "input": "The cat chased the mouse through the garden and finally caught it near the pond.",
        "output": "The cat caught the mouse near the pond."
    },
    {
        "instruction": "Classify the sentiment of the text.",
        "input": "I hated the movie. It was too long and boring.",
        "output": "Negative"
    }
]

dataset = Dataset.from_list(examples)

In [ ]:
def preprocess(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")

    if input_text.strip():
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    target = example["output"]
    model_inputs = tokenizer(prompt, max_length=512, truncation=True)
    labels = tokenizer(target, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess)


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./finetuned-flan-t5",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
!pip install wandb --quiet


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


/tmp/ipython-input-21-3812423459.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss


TrainOutput(global_step=3, training_loss=0.9247190157572428, metrics={'train_runtime': 474.1123, 'train_samples_per_second': 0.019, 'train_steps_per_second': 0.006, 'total_flos': 397212761088.0, 'train_loss': 0.9247190157572428, 'epoch': 3.0})

In [ ]:
def test_model(instruction, input_text=""):
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try it
print(test_model("Translate the sentence into French.", "I am learning to code."))
print(test_model("Summarize the paragraph.", "The climate is changing rapidly. Polar ice caps are melting."))
print(test_model("Convert Fahrenheit to Celsius.", "98.6"))
print(test_model("Write a tweet promoting climate awareness."))
print(test_model("Paraphrase this sentence.", "The quick brown fox jumped over the lazy dog."))
print(test_model("Answer the question based on the paragraph.",
                 "The Amazon rainforest is the largest tropical rainforest in the world. It is home to millions of species of plants and animals. Question: What is the Amazon rainforest known for?"))
print(test_model("List three benefits of exercise."))



Je ai en cours d'apprentissage.
Warming is taking place.
98.6
@sad_sad_sad_sad_sad_sad_ i'm a little sad
The lazy dog was jumped over the fox.
millions of species of plants and animals
Getting more exercise : -more exercise -more people -fewer -more people
